# Modelo Supervisado 2: XGBoost

Este modelo complementa el análisis abordando el objetivo de **clasificar el nivel de riesgo agrícola** (multiclase) de una siembra.
Se utiliza **XGBoost (Extreme Gradient Boosting)** por su robustez ante datos tabulares no lineales y su capacidad para manejar clases desbalanceadas mediante pesos (`sample_weight`).

## Fase 0: Configuración Inicial y Carga de Datos

In [ ]:
import sys
sys.path.insert(0, '..')
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    f1_score, accuracy_score, log_loss
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from pathlib import Path

from src.data_loader import DataLoader
from src.xgboost_model import XGBoostDataPrep

# Configuraciones globales
RANDOM_STATE = 42
sns.set_theme(style='whitegrid', palette='muted')
Path('figuras').mkdir(exist_ok=True)

In [ ]:
# Carga de datos
loader = DataLoader('../data/raw/siap_2010_2024.csv')
try:
    df_raw = loader.cargar()
except FileNotFoundError:
    loader = DataLoader('../siap_procesado.csv')
    df_raw = loader.cargar()

print('Dimensiones del dataset:', df_raw.shape)
df_raw.dropna(subset=['Sembrada', 'Volumenproduccion'], inplace=True)

## Fase 1: Ingeniería de Features y Preparación de Datos

Se aplicará el preprocesamiento definido en `src/xgboost_model.py`:
1. Codificación de variables (LabelEncoding).
2. Creación de variables derivadas (`log_sembrada`, `interaccion_mod_ciclo`).
3. **Split Temporal**: 
   - Entrenamiento: 2010-2020
   - Validación: 2021-2022
   - Test: 2023-2024
4. Cálculo de pesos para el desbalance de clases.

In [ ]:
prep = XGBoostDataPrep(df_raw)
df_processed = prep.preprocess()

(X_train, y_train), (X_val, y_val), (X_test, y_test) = prep.temporal_split()

# Calcular sample weights para entrenamiento debido al desbalance extremo
sample_weights = prep.get_sample_weights(y_train)
target_names = list(prep.riesgo_map.keys())

## Fase 2: Modelo Baseline (XGBoost sin optimizar)

Entrenaremos un clasificador multiclase de XGBoost utilizando sus hiperparámetros por defecto para establecer una métrica de referencia (baseline).

In [ ]:
xgb_baseline = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=5,
    random_state=RANDOM_STATE,
    n_estimators=100,
    tree_method='hist'
)

print('Entrenando modelo Baseline...')
xgb_baseline.fit(
    X_train, y_train, 
    sample_weight=sample_weights,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=False
)
print('¡Modelo Baseline entrenado!')

In [ ]:
y_val_pred = xgb_baseline.predict(X_val)

print('=== Reporte de Clasificación (Baseline en Validación) ===')
print(classification_report(y_val, y_val_pred, target_names=target_names))

f1_macro_base = f1_score(y_val, y_val_pred, average='macro')
acc_base = accuracy_score(y_val, y_val_pred)
print(f'F1-Score (Macro): {f1_macro_base:.4f}')
print(f'Accuracy: {acc_base:.4f}')

In [ ]:
cm_base = confusion_matrix(y_val, y_val_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_base, annot=True, fmt='d', cmap='Blues', 
            xticklabels=target_names, yticklabels=target_names, ax=ax)
ax.set_title('Matriz de Confusión - Modelo Baseline')
ax.set_ylabel('Valor Real')
ax.set_xlabel('Predicción')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('figuras/confusion_matrix_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

## Fase 3: Regularización con Gamma, Lambda (L2) y Alpha (L1)

XGBoost puede sobreajustarse a los datos de entrenamiento. Para contrarrestar esto, analizaremos el efecto de tres hiperparámetros de regularización:

- **Gamma ($\gamma$, `min_split_loss`)**: Reducción mínima de pérdida requerida para hacer una partición adicional en un nodo hoja. A mayor gamma, el algoritmo se vuelve más conservador, podando ramas que no aportan ganancia suficiente.

$$\text{Ganancia} = \frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L + G_R)^2}{H_L + H_R + \lambda} - \gamma$$

- **Lambda ($\lambda$, `reg_lambda`, L2)**: Regularización L2 sobre los pesos de las hojas. Suaviza las predicciones reduciendo la magnitud cuadrada de los pesos, lo que disminuye la varianza del modelo.

- **Alpha ($\alpha$, `reg_alpha`, L1)**: Regularización L1 sobre los pesos de las hojas. Puede llevar algunos pesos exactamente a cero, produciendo modelos más simples (*sparsity*).

### Experimento 1: Efecto individual de Gamma

In [ ]:
# Valores de gamma a explorar
gamma_values = [0, 0.1, 0.5, 1, 2, 5, 10]

gamma_results = []
for g in gamma_values:
    model = xgb.XGBClassifier(
        objective='multi:softprob', num_class=5,
        random_state=RANDOM_STATE, n_estimators=100,
        tree_method='hist',
        gamma=g, reg_lambda=1, reg_alpha=0  # defaults para lambda y alpha
    )
    model.fit(X_train, y_train, sample_weight=sample_weights,
              eval_set=[(X_val, y_val)], verbose=False)
    
    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val)
    
    gamma_results.append({
        'gamma': g,
        'f1_macro': f1_score(y_val, y_pred, average='macro'),
        'accuracy': accuracy_score(y_val, y_pred),
        'logloss': log_loss(y_val, y_proba)
    })
    print(f'  gamma={g:>5} → F1-macro={gamma_results[-1]["f1_macro"]:.4f}  logloss={gamma_results[-1]["logloss"]:.4f}')

df_gamma = pd.DataFrame(gamma_results)
print('\nResultados completos:')
display(df_gamma)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(df_gamma['gamma'], df_gamma['f1_macro'], marker='o', color='#3b7fc4', linewidth=2)
axes[0].set_title('Efecto de Gamma en F1-Macro', fontsize=13)
axes[0].set_xlabel('Gamma ($\gamma$)')
axes[0].set_ylabel('F1-Score (Macro)')
axes[0].axhline(y=f1_macro_base, color='red', linestyle='--', label=f'Baseline ({f1_macro_base:.4f})')
axes[0].legend()

axes[1].plot(df_gamma['gamma'], df_gamma['logloss'], marker='s', color='#e07b39', linewidth=2)
axes[1].set_title('Efecto de Gamma en Log Loss', fontsize=13)
axes[1].set_xlabel('Gamma ($\gamma$)')
axes[1].set_ylabel('Log Loss')

plt.tight_layout()
plt.savefig('figuras/exp1_gamma.png', dpi=150, bbox_inches='tight')
plt.show()

best_gamma = df_gamma.loc[df_gamma['f1_macro'].idxmax(), 'gamma']
print(f'\nMejor gamma encontrado: {best_gamma}')

### Experimento 2: Efecto individual de Lambda (L2)

In [ ]:
lambda_values = [0, 0.01, 0.1, 1, 5, 10, 50, 100]

lambda_results = []
for l in lambda_values:
    model = xgb.XGBClassifier(
        objective='multi:softprob', num_class=5,
        random_state=RANDOM_STATE, n_estimators=100,
        tree_method='hist',
        gamma=best_gamma, reg_lambda=l, reg_alpha=0
    )
    model.fit(X_train, y_train, sample_weight=sample_weights,
              eval_set=[(X_val, y_val)], verbose=False)
    
    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val)
    
    lambda_results.append({
        'reg_lambda': l,
        'f1_macro': f1_score(y_val, y_pred, average='macro'),
        'accuracy': accuracy_score(y_val, y_pred),
        'logloss': log_loss(y_val, y_proba)
    })
    print(f'  lambda={l:>6} → F1-macro={lambda_results[-1]["f1_macro"]:.4f}  logloss={lambda_results[-1]["logloss"]:.4f}')

df_lambda = pd.DataFrame(lambda_results)
print('\nResultados completos:')
display(df_lambda)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(df_lambda['reg_lambda'], df_lambda['f1_macro'], marker='o', color='#3b7fc4', linewidth=2)
axes[0].set_title('Efecto de Lambda (L2) en F1-Macro', fontsize=13)
axes[0].set_xlabel('Lambda ($\lambda$)')
axes[0].set_ylabel('F1-Score (Macro)')
axes[0].set_xscale('symlog', linthresh=0.1)
axes[0].axhline(y=f1_macro_base, color='red', linestyle='--', label=f'Baseline ({f1_macro_base:.4f})')
axes[0].legend()

axes[1].plot(df_lambda['reg_lambda'], df_lambda['logloss'], marker='s', color='#e07b39', linewidth=2)
axes[1].set_title('Efecto de Lambda (L2) en Log Loss', fontsize=13)
axes[1].set_xlabel('Lambda ($\lambda$)')
axes[1].set_ylabel('Log Loss')
axes[1].set_xscale('symlog', linthresh=0.1)

plt.tight_layout()
plt.savefig('figuras/exp2_lambda.png', dpi=150, bbox_inches='tight')
plt.show()

best_lambda = df_lambda.loc[df_lambda['f1_macro'].idxmax(), 'reg_lambda']
print(f'\nMejor lambda encontrado: {best_lambda}')

### Experimento 3: Efecto individual de Alpha (L1)

In [ ]:
alpha_values = [0, 0.001, 0.01, 0.1, 1, 5, 10, 50]

alpha_results = []
for a in alpha_values:
    model = xgb.XGBClassifier(
        objective='multi:softprob', num_class=5,
        random_state=RANDOM_STATE, n_estimators=100,
        tree_method='hist',
        gamma=best_gamma, reg_lambda=best_lambda, reg_alpha=a
    )
    model.fit(X_train, y_train, sample_weight=sample_weights,
              eval_set=[(X_val, y_val)], verbose=False)
    
    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val)
    
    alpha_results.append({
        'reg_alpha': a,
        'f1_macro': f1_score(y_val, y_pred, average='macro'),
        'accuracy': accuracy_score(y_val, y_pred),
        'logloss': log_loss(y_val, y_proba)
    })
    print(f'  alpha={a:>6} → F1-macro={alpha_results[-1]["f1_macro"]:.4f}  logloss={alpha_results[-1]["logloss"]:.4f}')

df_alpha = pd.DataFrame(alpha_results)
print('\nResultados completos:')
display(df_alpha)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(df_alpha['reg_alpha'], df_alpha['f1_macro'], marker='o', color='#3b7fc4', linewidth=2)
axes[0].set_title('Efecto de Alpha (L1) en F1-Macro', fontsize=13)
axes[0].set_xlabel('Alpha ($\\alpha$)')
axes[0].set_ylabel('F1-Score (Macro)')
axes[0].set_xscale('symlog', linthresh=0.001)
axes[0].axhline(y=f1_macro_base, color='red', linestyle='--', label=f'Baseline ({f1_macro_base:.4f})')
axes[0].legend()

axes[1].plot(df_alpha['reg_alpha'], df_alpha['logloss'], marker='s', color='#e07b39', linewidth=2)
axes[1].set_title('Efecto de Alpha (L1) en Log Loss', fontsize=13)
axes[1].set_xlabel('Alpha ($\\alpha$)')
axes[1].set_ylabel('Log Loss')
axes[1].set_xscale('symlog', linthresh=0.001)

plt.tight_layout()
plt.savefig('figuras/exp3_alpha.png', dpi=150, bbox_inches='tight')
plt.show()

best_alpha = df_alpha.loc[df_alpha['f1_macro'].idxmax(), 'reg_alpha']
print(f'\nMejor alpha encontrado: {best_alpha}')

### Experimento 4: Búsqueda conjunta (RandomizedSearchCV)

Realizamos una búsqueda aleatoria combinando gamma, lambda, alpha junto con otros hiperparámetros clave del modelo.

In [ ]:
param_dist = {
    'gamma': [0, 0.1, 0.5, 1, 2, 5],
    'reg_lambda': [0.01, 0.1, 1, 5, 10, 50],
    'reg_alpha': [0, 0.01, 0.1, 1, 5, 10],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [100, 200, 300],
    'min_child_weight': [1, 3, 5, 10],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0]
}

xgb_search = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=5,
    random_state=RANDOM_STATE,
    tree_method='hist'
)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

random_search = RandomizedSearchCV(
    estimator=xgb_search,
    param_distributions=param_dist,
    n_iter=50,
    scoring='f1_macro',
    cv=cv,
    n_jobs=-1,
    verbose=1,
    random_state=RANDOM_STATE,
    return_train_score=True
)

print('Iniciando búsqueda de hiperparámetros (esto puede tardar varios minutos)...')
random_search.fit(X_train, y_train, sample_weight=sample_weights)

print(f'\nMejor F1-Macro CV: {random_search.best_score_:.4f}')
print(f'Mejores hiperparámetros:')
for param, val in random_search.best_params_.items():
    print(f'  {param}: {val}')

In [ ]:
# Tabla resumen de los top 10 mejores resultados de la búsqueda
cv_results = pd.DataFrame(random_search.cv_results_)
cols_show = ['rank_test_score', 'mean_test_score', 'std_test_score',
             'param_gamma', 'param_reg_lambda', 'param_reg_alpha',
             'param_max_depth', 'param_learning_rate', 'param_n_estimators']
top10 = cv_results[cols_show].sort_values('rank_test_score').head(10)
print('=== Top 10 configuraciones ===')
display(top10)

### Tabla comparativa de los 4 experimentos

Resumen del efecto de la regularización sobre el desempeño del modelo.

In [ ]:
# Resumen comparativo
resumen = pd.DataFrame({
    'Configuración': ['Baseline (defaults)', 
                      f'Mejor Gamma (γ={best_gamma})',
                      f'Mejor Lambda (λ={best_lambda})',
                      f'Mejor Alpha (α={best_alpha})',
                      'RandomizedSearchCV (conjunto)'],
    'F1-Macro': [
        f1_macro_base,
        df_gamma.loc[df_gamma['f1_macro'].idxmax(), 'f1_macro'],
        df_lambda.loc[df_lambda['f1_macro'].idxmax(), 'f1_macro'],
        df_alpha.loc[df_alpha['f1_macro'].idxmax(), 'f1_macro'],
        random_search.best_score_
    ],
    'Log Loss': [
        log_loss(y_val, xgb_baseline.predict_proba(X_val)),
        df_gamma.loc[df_gamma['f1_macro'].idxmax(), 'logloss'],
        df_lambda.loc[df_lambda['f1_macro'].idxmax(), 'logloss'],
        df_alpha.loc[df_alpha['f1_macro'].idxmax(), 'logloss'],
        np.nan  # CV no reporta logloss directamente
    ]
})

print('=== Resumen Comparativo de Regularización ===')
display(resumen)